<a href="https://colab.research.google.com/github/yorique/colab/blob/main/CHIRPSnativo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
import ee
ee.Authenticate()

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import importlib.util
import json

ROOT = Path("/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2")
DEST = ROOT / "Resultados Finais" / "CHIRPS Nativo"
terms = ("bloco 26", "bloco 3", "bloco 2", "chirps", "bacia", "limite", "serie_mensal")

print("ROOT_EXISTS", ROOT.is_dir())
print("DEST_EXISTS", DEST.is_dir())
print("EARTHENGINE_API_INSTALLED", importlib.util.find_spec("ee") is not None)

hits = []
for p in ROOT.rglob("*"):
    if not p.is_file():
        continue
    name = p.name.lower()
    if any(term in name for term in terms):
        hits.append(p)
print("CANDIDATE_FILES", len(hits))
for p in hits[:250]:
    print("FILE", p, "SIZE", p.stat().st_size)

ROOT_EXISTS True
DEST_EXISTS True
EARTHENGINE_API_INSTALLED True
CANDIDATE_FILES 318
FILE /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco3_V2_CHIRPS_Mensal_Caete.json SIZE 2913
FILE /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco7_V2_Agregacao_Grade_CHIRPS_Caete.json SIZE 6444
FILE /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco14_V2_FaseB_Multibacias.json SIZE 2686
FILE /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco15_V2_Grades_CHIRPS_Lote.json SIZE 3438
FILE /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco19_V2_Agregacao_CHIRPS.json SIZE 2819
FILE /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Configurações/Configuracao_Bloco21_V2_Modelagem_Multibacias.json SIZE 3386
FILE /content/drive/MyDrive/Meus arquivos/UFRA/G

In [3]:
import csv, json, importlib.util
from pathlib import Path

def show_json(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    print("JSON", path.name, json.dumps(data, ensure_ascii=False, indent=2))
    return data

def header_and_count(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f, delimiter=";")
        header = next(reader)
        count = sum(1 for _ in reader)
    return header, count

cfg_b3 = ROOT / "Configurações" / "Configuracao_Bloco3_V2_CHIRPS_Mensal_Caete.json"
cfg_b15 = ROOT / "Configurações" / "Configuracao_Bloco15_V2_Grades_CHIRPS_Lote.json"
show_json(cfg_b3)
show_json(cfg_b15)

limits = ROOT / "Limites" / "Limites_11_Bacias_2024_EPSG31983.gpkg"
grid = ROOT / "Artefatos Operacionais" / "Fase B - Grades e CHIRPS" / "Grade_CHIRPS_10_Bacias_FaseB.gpkg"
reference = ROOT / "Resultados Finais" / "Extracao Temporal 11 Bacias - Bloco 26 V2" / "Tabelas" / "Serie_Mensal_Bacia_11_Bacias_Bloco26_V2.csv"
if not reference.is_file():
    reference = ROOT / "Auditorias" / "Bloco 26" / "Serie_Mensal_Bacia_11_Bacias_Bloco26_V2.csv"
print("LIMITS", limits, limits.is_file(), limits.stat().st_size if limits.is_file() else None)
print("GRID", grid, grid.is_file(), grid.stat().st_size if grid.is_file() else None)
print("REFERENCE", reference, reference.is_file())
if reference.is_file():
    print("REFERENCE_HEADER_COUNT", header_and_count(reference))
print("DEST_CONTENTS", [(p.name, p.stat().st_size) for p in sorted(DEST.iterdir())] if DEST.is_dir() else None)

try:
    import geopandas as gpd
    print("GEOPANDAS", gpd.__version__)
    b = gpd.read_file(limits)
    print("LIMITS_CRS", b.crs, "ROWS", len(b), "COLUMNS", list(b.columns))
    print("LIMITS_RECORDS", b.drop(columns="geometry").to_dict(orient="records"))
except Exception as exc:
    print("GEOPANDAS_ERROR", type(exc).__name__, str(exc))

try:
    import ee
    ee.Initialize()
    info = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").limit(1).getInfo()
    print("EE_OK", bool(info), "FIRST_ID", info["features"][0]["id"] if info and info.get("features") else None)
except Exception as exc:
    print("EE_ERROR", type(exc).__name__, str(exc))

JSON Configuracao_Bloco3_V2_CHIRPS_Mensal_Caete.json {
  "bloco": "BLOCO_3_V2",
  "equivalencia_amcdm": "D2_V2",
  "versao": "BLOCO_3_V2_v1.0.0",
  "data_utc": "2026-08-15T19:52:46+00:00",
  "projeto": "CHIRPS_Downscale_V2_11_Bacias_2000_2024",
  "bacia_piloto": "caete",
  "fonte": {
    "colecao": "UCSB-CHG/CHIRPS/DAILY",
    "banda": "precipitation",
    "unidade_diaria": "mm_dia",
    "operacao_mensal": "soma",
    "unidade_mensal": "mm_mes",
    "suavizacao": false
  },
  "periodo": {
    "ano_inicial": 2000,
    "ano_final": 2024,
    "meses_esperados": 300,
    "meses_construidos": 300,
    "meses_completos": 300,
    "meses_incompletos": 0,
    "dias_esperados": 9132,
    "imagens_diarias_usadas": 9132
  },
  "grade_chirps": {
    "crs": "EPSG:4326",
    "transform": [
      0.05,
      0.0,
      -180.0,
      0.0,
      -0.05,
      50.0
    ],
    "resolucao_x_graus": 0.05,
    "resolucao_y_graus": 0.05,
    "escala_nominal_m": 5565.974539663679,
    "criterio": "INTERSECTANT

In [7]:
import ee
try:
    ee.Initialize(project="projecttcc-489901")
    probe = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").limit(1).getInfo()
    print("EE_AUTHENTICATED", bool(probe), probe["features"][0]["id"] if probe and probe.get("features") else None)
except Exception as exc:
    print("EE_AUTH_ERROR", type(exc).__name__, str(exc))

EE_AUTHENTICATED True UCSB-CHG/CHIRPS/DAILY/19810101


In [8]:
import pandas as pd
import geopandas as gpd

paths = {
    "caete_cells": ROOT / "Artefatos Operacionais" / "CHIRPS Alvo Grosso" / "Tabela_CHIRPS_Celula_Mes_Caete_2000_2024.csv",
    "other_cells": ROOT / "Artefatos Operacionais" / "Fase B - Grades e CHIRPS" / "Tabela_CHIRPS_Celula_Mes_10_Bacias_2000_2024.csv",
    "caete_grid": ROOT / "Artefatos Operacionais" / "Grades de Referência" / "Grade_CHIRPS_Caete_Bloco2_V2.gpkg",
    "other_grid": ROOT / "Artefatos Operacionais" / "Fase B - Grades e CHIRPS" / "Grade_CHIRPS_10_Bacias_FaseB.gpkg",
}
for label, path in paths.items():
    print(label, path.is_file(), path)
    if path.suffix == ".csv":
        df = pd.read_csv(path, sep=";", nrows=3)
        print("COLUMNS", list(df.columns))
        print(df.to_dict(orient="records"))
    else:
        gdf = gpd.read_file(path)
        print("CRS", gdf.crs, "ROWS", len(gdf), "COLUMNS", list(gdf.columns))
        print(gdf.drop(columns="geometry").head(3).to_dict(orient="records"))

caete_cells True /content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Artefatos Operacionais/CHIRPS Alvo Grosso/Tabela_CHIRPS_Celula_Mes_Caete_2000_2024.csv
COLUMNS ['Bacia', 'cell_id', 'chirps_row', 'chirps_col', 'centro_lon', 'centro_lat', 'centro_na_bacia', 'area_celula_km2', 'area_na_bacia_km2', 'fracao_bacia', 'Ano', 'Mes', 'Ano_Mes', 'Precipitacao_CHIRPS_mm', 'Unidade', 'Fonte', 'Banda_Origem', 'Resolucao_Origem', 'Criterio_Grade']
[{'Bacia': 'caete', 'cell_id': 'CHIRPS_-4672500_-92500', 'chirps_row': 1018, 'chirps_col': 2665, 'centro_lon': -46.725, 'centro_lat': -0.9250000000000056, 'centro_na_bacia': False, 'area_celula_km2': 30.77223324204864, 'area_na_bacia_km2': 0.3091837527497295, 'fracao_bacia': 0.0100474915264598, 'Ano': 2000, 'Mes': 1, 'Ano_Mes': '2000-01', 'Precipitacao_CHIRPS_mm': 226.7188873291016, 'Unidade': 'mm_mes', 'Fonte': 'UCSB-CHG/CHIRPS/DAILY', 'Banda_Origem': 'precipitation', 'Resolucao_Origem': '0.05_grau', 'Criterio_Grade': 'INTERSECTANTE'}, {'Ba

In [9]:
import numpy as np
import pandas as pd
import geopandas as gpd
import ee

limits_gdf = gpd.read_file(limits).to_crs("EPSG:31983")[["bacia", "geometry"]]
boundary_by_basin = dict(zip(limits_gdf["bacia"], limits_gdf.geometry))

caete_grid_gdf = gpd.read_file(paths["caete_grid"])
caete_grid_gdf["Bacia"] = "caete"
other_grid_gdf = gpd.read_file(paths["other_grid"]).rename(columns={"Bacia": "Bacia"})
all_grid = pd.concat([caete_grid_gdf, other_grid_gdf], ignore_index=True)
all_grid = gpd.GeoDataFrame(all_grid, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:31983")

all_grid["area_intersecao_recalculada_km2"] = [
    geom.intersection(boundary_by_basin[basin]).area / 1_000_000
    for basin, geom in zip(all_grid["Bacia"], all_grid.geometry)
]
all_grid["dif_area_km2"] = all_grid["area_intersecao_recalculada_km2"] - all_grid["area_na_bacia_km2"]
area_check = all_grid.groupby("Bacia").agg(
    celulas_intersectantes=("cell_id", "size"),
    area_tabela_km2=("area_na_bacia_km2", "sum"),
    area_recalculada_km2=("area_intersecao_recalculada_km2", "sum"),
    max_abs_dif_celula_km2=("dif_area_km2", lambda s: float(np.abs(s).max())),
).reset_index()
print(area_check.to_string(index=False))

caete_cell_data = pd.read_csv(paths["caete_cells"], sep=";")
other_cell_data = pd.read_csv(paths["other_cells"], sep=";")
raw_cells = pd.concat([
    caete_cell_data[["Bacia", "cell_id", "Ano", "Mes", "Ano_Mes", "Precipitacao_CHIRPS_mm", "Unidade", "Fonte", "Banda_Origem", "Resolucao_Origem", "Criterio_Grade"]],
    other_cell_data[["Bacia", "cell_id", "Ano", "Mes", "Ano_Mes", "Precipitacao_CHIRPS_mm", "Unidade", "Fonte", "Banda_Origem", "Resolucao_Origem", "Criterio_Grade"]],
], ignore_index=True)
print("RAW_ROWS", len(raw_cells), "UNIQUE_BASIN_MONTH", raw_cells[["Bacia","Ano_Mes"]].drop_duplicates().shape[0])
print("DUPLICATE_BASIN_CELL_MONTH", int(raw_cells.duplicated(["Bacia","cell_id","Ano_Mes"]).sum()))
print("NEGATIVE_VALUES", int((raw_cells["Precipitacao_CHIRPS_mm"] < 0).sum()), "MISSING_VALUES", int(raw_cells["Precipitacao_CHIRPS_mm"].isna().sum()))

probe_row = raw_cells[(raw_cells["Bacia"] == "caete") & (raw_cells["Ano"] == 2000) & (raw_cells["Mes"] == 1)].iloc[0]
monthly_probe = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate("2000-01-01", "2000-02-01").select("precipitation").sum()
ee_value = monthly_probe.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point([float(caete_grid_gdf.iloc[0]["centro_lon"]), float(caete_grid_gdf.iloc[0]["centro_lat"])]),
    scale=5565.974539663679,
    crs="EPSG:4326",
).get("precipitation").getInfo()
print("EE_PROBE", {"cell_id": probe_row["cell_id"], "stored_mm": float(probe_row["Precipitacao_CHIRPS_mm"]), "ee_mm": ee_value, "abs_diff": abs(float(probe_row["Precipitacao_CHIRPS_mm"]) - float(ee_value))})

    Bacia  celulas_intersectantes  area_tabela_km2  area_recalculada_km2  max_abs_dif_celula_km2
     apeu                      48       917.729801            917.729801                     0.0
    caete                     103      2199.495905           2199.495905                     0.0
   caripi                      47       842.900341            842.900341                     0.0
    guama                     101      2138.517481           2138.517481                     0.0
 maracana                     129      2983.890033           2983.890033                     0.0
marapanim                      93      2047.067160           2047.067160                     0.0
  morcego                      68      1134.080110           1134.080110                     0.0
    piria                      87      1791.083406           1791.083406                     0.0
quatipuru                      48       950.704992            950.704992                     0.0
 taperacu                     

In [10]:
import hashlib
import json
import numpy as np
import pandas as pd

def sha256_path(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

OUT_CSV = DEST / "Serie_Mensal_CHIRPS_Nativo_11_Bacias_2000_2024.csv"
OUT_REPORT = DEST / "Relatorio_Execucao_CHIRPS_Nativo_11_Bacias_2000_2024.md"
OUT_MANIFEST = DEST / "Manifesto_Integridade_CHIRPS_Nativo_11_Bacias_2000_2024.json"
assert not any(p.exists() for p in (OUT_CSV, OUT_REPORT, OUT_MANIFEST)), "Destino já contém um dos arquivos propostos; não sobrescrever."

weights = pd.DataFrame(all_grid.drop(columns="geometry"))[
    ["Bacia", "cell_id", "area_na_bacia_km2", "area_intersecao_recalculada_km2"]
].rename(columns={"area_intersecao_recalculada_km2": "Peso_Area_km2"})
raw = raw_cells.merge(weights, on=["Bacia", "cell_id"], how="left", validate="many_to_one")
assert raw["Peso_Area_km2"].notna().all(), "Há valores CHIRPS sem peso de interseção."
assert (raw["Peso_Area_km2"] > 0).all(), "Há pesos de área não positivos."

area_total = weights.groupby("Bacia", as_index=False)["Peso_Area_km2"].sum().rename(columns={"Peso_Area_km2": "Area_Bacia_km2"})
support = raw[["Bacia", "cell_id", "Peso_Area_km2"]].drop_duplicates()
area_supported = support.groupby("Bacia", as_index=False).agg(
    Celulas_CHIRPS_Validas=("cell_id", "size"),
    Area_Valida_km2=("Peso_Area_km2", "sum"),
)
area_all = weights.groupby("Bacia", as_index=False).agg(Celulas_Intersectantes=("cell_id", "size"))
coverage = area_all.merge(area_supported, on="Bacia").merge(area_total, on="Bacia")
coverage["Celulas_Sem_Suporte"] = coverage["Celulas_Intersectantes"] - coverage["Celulas_CHIRPS_Validas"]
coverage["Area_Representada_pct"] = 100 * coverage["Area_Valida_km2"] / coverage["Area_Bacia_km2"]

def weighted_std(values, weights):
    mean = np.average(values, weights=weights)
    return np.sqrt(np.average((values - mean) ** 2, weights=weights))

out_rows = []
for (basin, year, month, ano_mes), group in raw.groupby(["Bacia", "Ano", "Mes", "Ano_Mes"], sort=True):
    vals = group["Precipitacao_CHIRPS_mm"].to_numpy(dtype=float)
    w = group["Peso_Area_km2"].to_numpy(dtype=float)
    c = coverage.loc[coverage["Bacia"].eq(basin)].iloc[0]
    quality = "SUPORTE_COMPLETO_300_300" if int(c["Celulas_Sem_Suporte"]) == 0 else "SUPORTE_PARCIAL_{}_CELULAS_SEM_SERIE_HISTORICA".format(int(c["Celulas_Sem_Suporte"]))
    out_rows.append({
        "Bacia": basin.upper(), "Fase": "CHIRPS_NATIVO", "Ano": int(year), "Mes": int(month),
        "Data": "{:04d}-{:02d}-01".format(int(year), int(month)), "Ano_Mes": ano_mes, "Unidade": "mm_mes",
        "Pixels_Validos": int(len(group)), "Area_Valida_km2": float(c["Area_Valida_km2"]),
        "Media_mm": float(np.average(vals, weights=w)), "Desvio_Padrao_Espacial_mm": float(weighted_std(vals, w)),
        "Minimo_mm": float(np.min(vals)), "P01_mm": float(np.quantile(vals, 0.01)),
        "P25_mm": float(np.quantile(vals, 0.25)), "Mediana_mm": float(np.quantile(vals, 0.50)),
        "P75_mm": float(np.quantile(vals, 0.75)), "P99_mm": float(np.quantile(vals, 0.99)),
        "Maximo_mm": float(np.max(vals)), "Arquivo_Fonte": "UCSB-CHG/CHIRPS/DAILY",
        "Banda_Fonte": "precipitation", "Excecao_Estrutural_Herdada": False,
        "Tipo_Excecao_Estrutural": "", "Restricoes_CHIRPS_Relaxadas": False,
        "Restricoes_Relaxadas_Cell_IDs": "", "Area_Restricoes_Relaxadas_km2": 0.0,
        "Area_Restricoes_Relaxadas_pct": 0.0, "Tratamento_Excecao_Estrutural": "SEM_PREENCHIMENTO_DE_NODATA",
        "Origem_Excecao_Estrutural": quality, "Media_Simples_mm": float(np.mean(vals)),
        "Celulas_Intersectantes": int(c["Celulas_Intersectantes"]), "Celulas_Sem_Suporte": int(c["Celulas_Sem_Suporte"]),
        "Area_Bacia_km2": float(c["Area_Bacia_km2"]), "Area_Representada_pct": float(c["Area_Representada_pct"]),
        "Metodo_Agregacao": "MEDIA_PONDERADA_POR_AREA_DE_INTERSECAO", "Resolucao_CHIRPS": "0.05_grau",
    })

out = pd.DataFrame(out_rows).sort_values(["Bacia", "Ano", "Mes"]).reset_index(drop=True)
expected_basins = sorted(limits_gdf["bacia"].str.upper().tolist())
assert len(out) == 3300, "Esperado 3300, obtido {}".format(len(out))
assert sorted(out["Bacia"].unique()) == expected_basins, "Conjunto de bacias divergente."
assert (out.groupby("Bacia").size() == 300).all(), "Alguma bacia não possui 300 meses."
assert out[["Bacia", "Ano", "Mes"]].duplicated().sum() == 0, "Há duplicatas."
assert out["Media_mm"].notna().all() and (out["Media_mm"] >= 0).all(), "Há precipitação inválida."
assert out["Data"].eq(out["Ano_Mes"] + "-01").all(), "Datas inconsistentes."

out.to_csv(OUT_CSV, sep=";", index=False, encoding="utf-8-sig", float_format="%.10f")
output_hash = sha256_path(OUT_CSV)
input_hashes = {
    "limites_11_bacias": sha256_path(limits),
    "grade_caete": sha256_path(paths["caete_grid"]),
    "grade_10_bacias": sha256_path(paths["other_grid"]),
    "chirps_celula_mes_caete": sha256_path(paths["caete_cells"]),
    "chirps_celula_mes_10_bacias": sha256_path(paths["other_cells"]),
    "referencia_bloco26": sha256_path(reference),
}
manifest = {
    "produto": OUT_CSV.name, "sha256_produto": output_hash, "registros": int(len(out)),
    "bacias": expected_basins, "periodo": {"inicio": "2000-01", "fim": "2024-12", "meses_por_bacia": 300},
    "fonte": {"colecao": "UCSB-CHG/CHIRPS/DAILY", "banda": "precipitation", "unidade": "mm_mes", "resolucao": "0.05_grau"},
    "metodo": "soma_diaria_mensal_historica_por_celula; media_mensal_ponderada_por_area_de_intersecao",
    "cobertura_por_bacia": coverage.to_dict(orient="records"), "hashes_entrada": input_hashes,
    "teste_ee": {"celula": str(probe_row["cell_id"]), "mes": "2000-01", "diferenca_absoluta_mm": abs(float(probe_row["Precipitacao_CHIRPS_mm"]) - float(ee_value))},
}
OUT_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

report = """# Relatório de execução — CHIRPS nativo, 11 bacias

## Resultado

- Produto: {}
- Registros: {} (11 bacias × 300 meses)
- Período: 2000-01 a 2024-12
- Unidade: mm/mês
- Fonte: UCSB-CHG/CHIRPS/DAILY, banda precipitation
- Agregação: média mensal ponderada pela área efetiva de interseção entre a célula CHIRPS nativa de 0,05° e cada limite oficial de bacia.
- Série de referência: Bloco 26 consultada somente para conferir a estrutura tabular; não foi usada como fonte de precipitação.

## Validações

- 3.300 registros, 300 por bacia.
- Sem duplicatas, valores ausentes ou precipitação negativa.
- Áreas de interseção recalculadas a partir dos limites oficiais: diferença máxima por célula igual a 0 km² em relação às grades históricas.
- Controle Earth Engine para a célula {} em 2000-01: diferença absoluta de {:.10f} mm entre a série histórica célula×mês e a consulta à coleção.
- Há quatro células com 0/300 meses de suporte histórico nas dez bacias da Fase B; elas não foram preenchidas com zero e a área efetivamente representada foi preservada nos campos de qualidade.

## Ressalva de suporte

A série produzida tem unidade bacia×ano×mês compatível com o Bloco 26, mas não tem o mesmo suporte espacial dos produtos AmCDM a 250 m: ela agrega diretamente células CHIRPS nativas por suas áreas de interseção com as bacias.
""".format(OUT_CSV.name, len(out), probe_row["cell_id"], abs(float(probe_row["Precipitacao_CHIRPS_mm"]) - float(ee_value)))
OUT_REPORT.write_text(report, encoding="utf-8")
print("OUTPUTS", {str(p): p.stat().st_size for p in (OUT_CSV, OUT_REPORT, OUT_MANIFEST)})
print("OUTPUT_SHA256", output_hash)
print("VALIDATION", {"records": len(out), "basins": out["Bacia"].nunique(), "months_per_basin": out.groupby("Bacia").size().to_dict(), "coverage": coverage.to_dict(orient="records")})

OUTPUTS {'/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Resultados Finais/CHIRPS Nativo/Serie_Mensal_CHIRPS_Nativo_11_Bacias_2000_2024.csv': 1446202, '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Resultados Finais/CHIRPS Nativo/Relatorio_Execucao_CHIRPS_Nativo_11_Bacias_2000_2024.md': 1463, '/content/drive/MyDrive/Meus arquivos/UFRA/GEE/TCC - CHIRPS - V2/Resultados Finais/CHIRPS Nativo/Manifesto_Integridade_CHIRPS_Nativo_11_Bacias_2000_2024.json': 4438}
OUTPUT_SHA256 ac455f77638696508f27af7294cb5af2ec7ed7cb14466a55261cb94bb9294998
VALIDATION {'records': 3300, 'basins': 11, 'months_per_basin': {'APEU': 300, 'CAETE': 300, 'CARIPI': 300, 'GUAMA': 300, 'MARACANA': 300, 'MARAPANIM': 300, 'MORCEGO': 300, 'PIRIA': 300, 'QUATIPURU': 300, 'TAPERACU': 300, 'URUMAJO': 300}, 'coverage': [{'Bacia': 'apeu', 'Celulas_Intersectantes': 48, 'Celulas_CHIRPS_Validas': 48, 'Area_Valida_km2': 917.7298014277184, 'Area_Bacia_km2': 917.7298014277184, 'Celulas_Sem_Suporte': 

In [11]:
saved = pd.read_csv(OUT_CSV, sep=";")
print("PERSISTENCE", OUT_CSV.is_file(), OUT_CSV.stat().st_size)
print("SAVED_SHA256", sha256_path(OUT_CSV))
print("SAVED_STRUCTURE", {"rows": len(saved), "columns": list(saved.columns), "basins": sorted(saved["Bacia"].unique()), "months_per_basin": saved.groupby("Bacia").size().to_dict(), "duplicates": int(saved.duplicated(["Bacia","Ano","Mes"]).sum()), "negative": int((saved["Media_mm"] < 0).sum()), "missing": int(saved["Media_mm"].isna().sum())})
print("AUXILIARY_PERSISTENCE", {p.name: p.is_file() for p in (OUT_REPORT, OUT_MANIFEST)})

PERSISTENCE True 1446202
SAVED_SHA256 ac455f77638696508f27af7294cb5af2ec7ed7cb14466a55261cb94bb9294998
SAVED_STRUCTURE {'rows': 3300, 'columns': ['Bacia', 'Fase', 'Ano', 'Mes', 'Data', 'Ano_Mes', 'Unidade', 'Pixels_Validos', 'Area_Valida_km2', 'Media_mm', 'Desvio_Padrao_Espacial_mm', 'Minimo_mm', 'P01_mm', 'P25_mm', 'Mediana_mm', 'P75_mm', 'P99_mm', 'Maximo_mm', 'Arquivo_Fonte', 'Banda_Fonte', 'Excecao_Estrutural_Herdada', 'Tipo_Excecao_Estrutural', 'Restricoes_CHIRPS_Relaxadas', 'Restricoes_Relaxadas_Cell_IDs', 'Area_Restricoes_Relaxadas_km2', 'Area_Restricoes_Relaxadas_pct', 'Tratamento_Excecao_Estrutural', 'Origem_Excecao_Estrutural', 'Media_Simples_mm', 'Celulas_Intersectantes', 'Celulas_Sem_Suporte', 'Area_Bacia_km2', 'Area_Representada_pct', 'Metodo_Agregacao', 'Resolucao_CHIRPS'], 'basins': ['APEU', 'CAETE', 'CARIPI', 'GUAMA', 'MARACANA', 'MARAPANIM', 'MORCEGO', 'PIRIA', 'QUATIPURU', 'TAPERACU', 'URUMAJO'], 'months_per_basin': {'APEU': 300, 'CAETE': 300, 'CARIPI': 300, 'GUAMA': 